In [1]:
from helper_functions import import_flight_data, extract_wind_data
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
data_dir = '/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data'
wind_dir = '/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/TWI data'

df = import_flight_data(data_dir=data_dir,limit=1, recursive=True)


In [3]:
COLUMN_MAP = {
    # time
    "Year": "year",
    "QAR YEAR": "year",
    "Date: Year (Derived)": "year",

    "MONTH": "month",
    "QAR MONTH": "month",
    "Date (month)": "month",

    "DAY": "day",
    "QAR DAY": "day",
    "Date (day)": "day",

    "GMT - Hours (BCD)": "hour",
    "QAR HOUR": "hour",
    "UTC Hours": "hour",

    "GMT - Minutes (BCD)": "minute",
    "QAR MINUTE": "minute",
    "UTC Minutes": "minute",

    "GMT Seconds": "second",
    "QAR SECOND": "second",
    "UTC Seconds": "second",

    # altitude
    "Radio Altitude": "radio_altitude",
    "Radio Height 1": "radio_altitude",

    # vertical acceleration
    "Vertical Acceleration": "vert_acc",
    "Normal acceleration": "vert_acc",
}

In [4]:
N_arrivals = 1

arrival_data = import_flight_data(data_dir=data_dir, 
    limit=N_arrivals, add_source_file=True)#, usecols=['Time (secs)','MONTH', 'DAY', 'Year', 'GMT - Hours (BCD)', 'GMT - Minutes (BCD)', 'GMT Seconds', 'Vertical Acceleration', 'Radio Altitude'])

#arrival_data.columns.tolist()

In [5]:
N_arrivals_tot = 3140
N_arrivals = N_arrivals_tot

arrival_data = import_flight_data(data_dir=data_dir,
    limit=N_arrivals, add_source_file=True, usecols=['month', 'day', 'year', 'hour', 'minute', 'second', 'vert_acc', 'radio_altitude'])




KeyboardInterrupt: 

In [6]:
from pathlib import Path
from typing import Optional, Union
import numpy as np

import pandas as pd

COLUMN_MAP = {
    # time
    "Year": "year",
    "QAR YEAR": "year",
    "Date: Year (Derived)": "year",

    "MONTH": "month",
    "QAR MONTH": "month",
    "Date (month)": "month",

    "DAY": "day",
    "QAR DAY": "day",
    "Date (day)": "day",

    "GMT - Hours (BCD)": "hour",
    "QAR HOUR": "hour",
    "UTC Hours": "hour",

    "GMT - Minutes (BCD)": "minute",
    "QAR MINUTE": "minute",
    "UTC Minutes": "minute",

    "GMT Seconds": "second",
    "QAR SECOND": "second",
    "UTC Seconds": "second",

    # altitude
    "Radio Altitude": "radio_altitude",
    "Radio Height 1": "radio_altitude",

    # vertical acceleration
    "Vertical Acceleration": "vert_acc",
    "Normal acceleration": "vert_acc",
}

def import_flight_data(
    data_dir: Optional[Union[str, Path]] = None,
    pattern: str = "*.csv",
    recursive: bool = True,
    start: Optional[int] = 0,
    limit: Optional[int] = None,
    add_source_file: bool = True,
    **read_csv_kwargs,
) -> pd.DataFrame:
    """
    Import CSV flight data from the turbulens data folder.

    Args:
        data_dir: Directory to search. Defaults to `<project_root>/data`.
        pattern: File match pattern, default is '*.csv'.
        recursive: If True, search subfolders recursively.
        start: Optional index to start loading files from.
        limit: Optional max number of files to load.
        add_source_file: If True, add a `source_file` column.
        **read_csv_kwargs: Extra keyword arguments passed to `pd.read_csv`.

    Returns:
        A concatenated pandas DataFrame containing all loaded CSV files.
    """
    root_data_dir = Path(__file__).resolve().parents[1] / "data"
    target_dir = Path(data_dir) if data_dir else root_data_dir

    finder = "rglob" if recursive else "glob"

    arrivals_dir = target_dir / "Arrivals"
    departures_dir = target_dir / "Departures"

    arrivals = sorted(getattr(arrivals_dir, finder)(pattern))
    departures = sorted(getattr(departures_dir, finder)(pattern))
    csv_files = arrivals + departures


    if limit is not None:
        csv_files = csv_files[start:start + limit]
    else:
        csv_files = csv_files[start:]

    if not csv_files:
        raise FileNotFoundError(
            f"No files found in '{target_dir}' using pattern '{pattern}'."
        )

    frames = []
    for csv_path in csv_files:
        if 'GKN' in csv_path.name: #Skip files with OY-GKN, as their format sucks
            continue
        frame = pd.read_csv(csv_path, low_memory=False)

        frame.columns = [COLUMN_MAP.get(c, c) for c in frame.columns]       
        needed = [
            'year','month','day',
            'hour','minute','second',
            'vert_acc','radio_altitude']      
          
        frame = frame[[c for c in needed if c in frame.columns]]        
        mask_time = np.isclose(frame['second'] % 1, 0) #!!!
        mask_alt = frame['radio_altitude'] <= 1500 #!!!
        frame = frame.loc[mask_time & mask_alt].reset_index(drop=True) #!!!

        if add_source_file:
            frame["source_file"] = str(csv_path)
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)

def extract_wind_data(flight_df, data_dir, offset=10): 
    """
    Extracts wind data from the TWI dataset for a given flight, based on the landing time and an offset for the recording start time.
    Args:
        flight_df (pd.DataFrame): A DataFrame containing data for one flight, including landing time information.
        data_dir (str): The directory containing the wind data files.
        offset (int): The number of minutes before the landing time to start recording wind data. Default is 10 minutes.
    Returns:
        pd.DataFrame: A DataFrame containing the wind data for the specified time range.
    """

    first = flight_df.iloc[0]

    month = first['month'].astype(int)
    day = first['day'].astype(int)
    if day < 10:
        day_string = '0'+str(day)
    else:
        day_string = str(day)

    if month < 10:
        month_string = '0'+str(month)
    else:        
        month_string = str(month)

    year = first['year'].astype(int)
    start_h = first['hour'].astype(int)
    start_m = first['minute'].astype(int)
    start_s = first['second'].astype(int)
    
    start_of_landing_time = pd.Timestamp(year=year,month=month,day=day,hour=start_h,minute=start_m,second=start_s)

    start_of_recording_time = start_of_landing_time - pd.Timedelta(minutes=offset)
    
    if start_of_recording_time.day != start_of_landing_time.day:
        raise ValueError("Start of recording time is on a different day than the landing time. Please adjust the offset or check the flight data.")
    
    date_string = str(year)+'-'+month_string+'-'+day_string
    file_string = 'TWI-'+date_string+'_UTC_log.csv'

    wind_df = pd.read_csv(data_dir + '/' + file_string, sep=';')

    wind_df['DateTime'] = pd.to_datetime(wind_df['DateTime'], format='%Y-%m-%d %H:%M:%S')

    mask_time = (wind_df['DateTime'] >= start_of_recording_time) & (wind_df['DateTime'] <= start_of_landing_time)

    wind_df = wind_df.loc[mask_time].reset_index(drop=True)
    wind_df['source_file'] = first['source_file']
    return wind_df

In [7]:
#full_wind_data = []
#idx = 0
#
#for flight_id, flight_df in arrival_data.groupby('source_file'):
#    idx +=1
#    print(f"Processing flight: {idx}/{len(arrival_data.groupby('source_file'))}")
#    try:
#        wind_df = extract_wind_data(flight_df, data_dir=wind_dir)
#        wind_df['source_file'] = flight_id
#        full_wind_data.append(wind_df)
#
#    except ValueError as e:
#        print(f"Error processing flight {flight_id}: {e}")
#full_wind_data = pd.concat(full_wind_data, ignore_index=True)
#full_wind_data.to_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject.csv', index=False)

In [10]:
all_wind_data = pd.read_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject/All_wind_data.csv')

In [ ]:
#['W04.VectorMeanWindSpeed', 'W04.VectorMeanWindDirection', 'W04.ScalarMaxWindSpeed', 
# 'W04.ScalarMaxWindAtDirection', 'W04.ScalarMeanWindSpeed', 'W04.ScalarWindSpeedDeviation', 
# 'W04.ScalarMeanWindDirection', 'W04.ScalarWindDirectionDeviation', 'W04.LastWindSpeed', 
# 'W04.LastWindDirection', 
# 'W22.VectorMeanWindSpeed', 'W22.VectorMeanWindDirection', 
# 'W22.ScalarMaxWindSpeed', 'W22.ScalarMaxWindAtDirection', 'W22.ScalarMeanWindSpeed', 
# 'W22.ScalarWindSpeedDeviation', 'W22.ScalarMeanWindDirection', 
# 'W22.ScalarWindDirectionDeviation', 'W22.LastWindSpeed', 'W22.LastWindDirection']

#columns_to_analyze = all_wind_data.columns.tolist()[8:-2]
#columns_to_analyze.remove('W04.OkPct')
#
#all_wind_data['u04'] = all_wind_data['W04.VectorMeanWindSpeed'] * np.cos(all_wind_data['W04.VectorMeanWindDirection'])
#all_wind_data['u22'] = all_wind_data['W22.VectorMeanWindSpeed'] * np.cos(all_wind_data['W22.VectorMeanWindDirection'])
#all_wind_data['v04'] = all_wind_data['W04.VectorMeanWindSpeed'] * np.sin(all_wind_data['W04.VectorMeanWindDirection'])
#all_wind_data['v22'] = all_wind_data['W22.VectorMeanWindSpeed'] * np.sin(all_wind_data['W22.VectorMeanWindDirection'])
#
#all_wind_data['du'] = all_wind_data['u04'] - all_wind_data['u22'] #Shear in u component
#all_wind_data['dv'] = all_wind_data['v04'] - all_wind_data['v22'] #Shear in v component
#
#columns_to_analyze += ['u04', 'v04', 'u22', 'v22', 'du', 'dv']
#
#columns_to_analyze.remove('W04.VectorMeanWindSpeed')
#columns_to_analyze.remove('W04.VectorMeanWindDirection')
#columns_to_analyze.remove('W22.VectorMeanWindSpeed')
#columns_to_analyze.remove('W22.VectorMeanWindDirection')
#
#print(columns_to_analyze)
#
#def p95(x):
#    return np.percentile(x, 95)
#
#def val_range(x):
#    return np.max(x) - np.min(x)
#
#def slope(x):
#    y = x.values
#    t = np.arange(len(y))
#    return np.polyfit(t, y, 1)[0]

#statistics = ['mean', 'std', p95, val_range, slope]
#
#
#all_wind_data['VectorMeanWindSpeed_diff'] = all_wind_data['W04.VectorMeanWindSpeed'] - all_wind_data['W22.ScalarMeanWindSpeed']
##all_wind_data['VectorMeanWindDirection_diff'] = all_wind_data['W04.VectorMeanWindDirection'] - all_wind_data['W22.ScalarMeanWindDirection']
#all_wind_data['ScalarMeanWindSpeed_diff'] = all_wind_data['W04.ScalarMeanWindSpeed'] - all_wind_data['W22.ScalarMeanWindSpeed']
#
#columns_to_analyze += ['VectorMeanWindSpeed_diff', 'ScalarMeanWindSpeed_diff']
#
#summary_df = (
#    all_wind_data
#    .groupby('source_file')[columns_to_analyze]
#    .agg(statistics))

['W04.ScalarMaxWindSpeed', 'W04.ScalarMaxWindAtDirection', 'W04.ScalarMeanWindSpeed', 'W04.ScalarWindSpeedDeviation', 'W04.ScalarMeanWindDirection', 'W04.ScalarWindDirectionDeviation', 'W04.LastWindSpeed', 'W04.LastWindDirection', 'W22.ScalarMaxWindSpeed', 'W22.ScalarMaxWindAtDirection', 'W22.ScalarMeanWindSpeed', 'W22.ScalarWindSpeedDeviation', 'W22.ScalarMeanWindDirection', 'W22.ScalarWindDirectionDeviation', 'W22.LastWindSpeed', 'W22.LastWindDirection', 'W22.OkPct', 'source_file', 'u04', 'v04', 'u22', 'v22', 'du', 'dv']


In [ ]:
summary_df.columns = [
    f"{col}_{stat}".replace('.', '_')
    for col, stat in summary_df.columns]

summary_df.to_csv("summary_2.csv", index=False)



In [24]:
all_wind_data = pd.read_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject/All_wind_data.csv')

In [ ]:
def add_uv(df, prefix):
    speed = df[f"{prefix}.VectorMeanWindSpeed"]
    direction = df[f"{prefix}.VectorMeanWindDirection"]

    df[f"{prefix}_u"] = speed * np.cos(direction)
    df[f"{prefix}_v"] = speed * np.sin(direction)

    return df

all_wind_data = add_uv(all_wind_data, 'W04')
all_wind_data = add_uv(all_wind_data, 'W22')

def slope(x):
    t = np.arange(len(x))
    return np.polyfit(t, x, 1)[0]


def p95(x):
    return np.percentile(x, 95)


def range_(x):
    return np.max(x) - np.min(x)


def mean_abs_change(x):
    return np.mean(np.abs(np.diff(x)))

def extract_features(group):
    feats = {}

    # --- vector stations ---
    for p in ["W04", "W22"]:
        for comp in ["u", "v"]:
            x = group[f"{p}_{comp}"].values

            feats[f"{p}_{comp}_mean"] = np.mean(x)
            feats[f"{p}_{comp}_std"] = np.std(x)
            feats[f"{p}_{comp}_p95"] = p95(x)
            feats[f"{p}_{comp}_range"] = range_(x)
            feats[f"{p}_{comp}_slope"] = slope(x)
            feats[f"{p}_{comp}_mac"] = mean_abs_change(x)

    # --- scalar wind speed features ---
    for p in ["W04", "W22"]:
        x = group[f"{p}.ScalarMeanWindSpeed"].values

        feats[f"{p}_speed_mean"] = np.mean(x)
        feats[f"{p}_speed_std"] = np.std(x)
        feats[f"{p}_speed_p95"] = p95(x)
        feats[f"{p}_speed_range"] = range_(x)
        feats[f"{p}_speed_slope"] = slope(x)

    # --- cross-station shear (IMPORTANT) ---
    w04_speed = group["W04.ScalarMeanWindSpeed"].values
    w22_speed = group["W22.ScalarMeanWindSpeed"].values
    diff = w04_speed - w22_speed
    feats["speed_shear_mean"] = np.mean(diff)
    feats["speed_shear_std"] = np.std(diff)
    feats["speed_shear_p95"] = p95(diff)
    feats["speed_shear_range"] = range_(diff)
    feats["speed_shear_slope"] = slope(diff)

    return pd.Series(feats)

feature_df = all_wind_data.groupby("source_file").apply(extract_features).reset_index()

/var/folders/yw/gbklf6sd5q3_sr89rbjcfv7w0000gn/T/ipykernel_20942/544393536.py:65: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  feature_df = all_wind_data.groupby("source_file").apply(extract_features).reset_index()


In [28]:
feature_df.drop(columns=["source_file"], inplace=True)
feature_df.to_csv("feature_df.csv", index=False)